# Oracle Infinity Recommendations

## Inventory de oportunidades de hotelaria internacional

Este notebook gera um CSV exatamente no layout fornecido pela ferramenta de Recommendations. Cada produto representa um hotel, identificado pelo `HotelId` presente na URL.

| Campo de saída | Regra |
|---|---|
| `Pr_ID` | `HotelId` extraído da URL |
| `Category1` | `hotel_internacional` |
| `Category2` | `PAIS_CIDADE` |
| `Price` | diária com desconto; usa diária normal somente como contingência |
| `Count` | `9999` |
| `Locale` | `pt-BR` |
| `Is_Del` | `0` — produto ativo |
| `Is_Rec` | `1` — produto recomendável |

## 1. Configuração e funções de apoio

Revise principalmente `OUTPUT_PATH` antes de executar. O notebook lê somente oportunidades com `is_active = true` e valida as colunas necessárias antes de transformar qualquer dado.

As funções auxiliares abaixo padronizam textos, interpretam datas em formatos comuns e convertem valores monetários brasileiros, como `1.234,56`, para números.

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Gera um CSV de Inventory compatível com Oracle Infinity Recommendations.
# Ajuste apenas as configuraçõeses abaixo antes de executar no Databricks.

import html

from pyspark.sql import DataFrame, Window, functions as F, types as T


# -----------------------------------------------------------------------------
# Configurações
# -----------------------------------------------------------------------------
SOURCE_TABLE = "cvc_planejamento_produtos_silver.tb_oportunidades_produtos_hotelaria_internacional"
OUTPUT_PATH = "dbfs:/FileStore/oracle_infinity_recommendations_inventory.csv"
IMAGE_URL = (
    "https://tools-images.services.cvc.com.br/resize?width=827&height=485&type=auto&quality=80&"
    "url=https%3A%2F%2Fi.travelapi.com%2Flodging%2F1000000%2F50000%2F41100%2F41081%2Feeb39ad6_z.jpg"
)

FAIL_ON_DUPLICATE_PRODUCT_IDS = False
CSV_PREVIEW_ROWS = 1_000  # Limite seguro para copiar o CSV durante a POC.

REQUIRED_SOURCE_COLUMNS = [
    "HOTEL",
    "DESTINO",
    "TIPO_DE_PRODUTO",
    "LINK_CVC_COM",
    "DIARIA_POR_PESSOA_DE_R",       
    "DIARIA_POR_PESSOA_POR_R",      
    "DESCONTO",
    "PAIS",
    "PERIODO_DE_EMBARQUE_INICIO_DD_MM_AAAA",
    "PERIODO_DE_EMBARQUE_FIM_DD_MM_AAAA",
]

OUTPUT_COLUMNS = [
    "Pr_ID", "Name", "Category1", "Category2", "Price", "Currency", "Page_URL",
    "Image_URL", "Count", "Locale", "Is_Del", "Is_Rec", "Thumbs_Image_URL",
]


def print_step(step: int, message: str) -> None:
    print(f"[{step}/7] {message}")


def assert_columns_exist(df: DataFrame, columns: list[str], context: str) -> None:
    available = {column.upper(): column for column in df.columns}
    missing = [column for column in columns if column.upper() not in available]
    if missing:
        raise ValueError(
            f"Colunas obrigatórias ausentes em {context}: {missing}. "
            f"Colunas disponíveis: {df.columns}"
        )

# Retorna o nome real da coluna, aceitando diferenças de maiúsculas/minúsculas
def source_column(df: DataFrame, expected_name: str) -> str:
    return next(column for column in df.columns if column.upper() == expected_name.upper())

# Converte nulos, NaN textual e espaços vazios para null
def clean_text(column: F.Column) -> F.Column:
    value = F.trim(column.cast("string"))
    return F.when(
        column.isNull() | F.lower(value).isin("", "nan", "null", "none", "nat"),
        F.lit(None).cast("string"),
    ).otherwise(value)

# Aceita os formatos mais comuns da origem, inclusive coluna já tipada como date
def parse_date(column: F.Column) -> F.Column:
    text = clean_text(column)
    return F.coalesce(
        F.to_date(F.try_to_timestamp(text, F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(text, F.lit("yyyy-MM-dd HH:mm:ss"))),
        F.to_date(F.try_to_timestamp(text, F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(text, F.lit("dd-MM-yyyy"))),
    )

# Converte valores BR (1.234,56) ou padrÃ£o (1234.56) para decimal
def parse_decimal(column: F.Column) -> F.Column:
    text = clean_text(column)
    normalized = F.when(
        text.rlike(r"^-?[0-9]{1,3}(\\.[0-9]{3})+,[0-9]+$"),
        F.regexp_replace(F.regexp_replace(text, r"\\.", ""), ",", "."),
    ).when(
        text.rlike(r"^-?[0-9]+,[0-9]+$"), F.regexp_replace(text, ",", ".")
    ).otherwise(F.regexp_replace(text, r"[^0-9.-]", ""))
    return normalized.cast(T.DecimalType(18, 2))

# Normalização estável para ca composição do Product ID
def normalize_for_id(column: F.Column) -> F.Column:
    text = clean_text(column)
    return F.upper(
        F.regexp_replace(
            F.regexp_replace(F.translate(text, "ÃÃ€Ã‚ÃƒÃ„Ã‰ÃˆÃŠÃ‹ÃÃŒÃŽÃÃ“Ã’Ã”Ã•Ã–ÃšÃ™Ã›ÃœÃ‡Ã‘", "AAAAAEEEEIIIIOOOOOUUUUCN"), r"[^A-Za-z0-9]+", "_"),
            r"(^_+|_+$)",
            "",
        )
    )


def load_data() -> DataFrame:
    print_step(2, "Executando query de origem...")
    query = f"""
        SELECT
            HOTEL,
            DESTINO,
            TIPO_DE_PRODUTO,
            LINK_CVC_COM,
            DIARIA_POR_PESSOA_DE_R,
            DIARIA_POR_PESSOA_POR_R,
            DESCONTO,
            PAIS,
            PERIODO_DE_EMBARQUE_INICIO_DD_MM_AAAA,
            PERIODO_DE_EMBARQUE_FIM_DD_MM_AAAA
        FROM {SOURCE_TABLE}
        WHERE is_active = true
          AND TRIM(LINK_CVC_COM) RLIKE '(?i)^https?://.+'
          AND UPPER(TRIM(VALIDO_PARA_SITE)) = 'SIM'
          AND UPPER(TRIM(U_N)) = 'CVC'
    """

    df = spark.sql(query)

    assert_columns_exist(df, REQUIRED_SOURCE_COLUMNS, SOURCE_TABLE)

    print(f"Registros encontrados: {df.count()}")
    print("Amostra dos dados retornados pela query:")
    df.show(10, truncate=False)

    return df



## 2. Normalização dos dados de origem

Nesta etapa, espaços desnecessários, valores vazios, `null` e `nan` são convertidos para nulo. As datas de início e fim são preparadas para o processamento mensal, e as duas colunas de diária recebem o tipo numérico adequado.

In [ ]:
# Padroniza textos, datas, ID do hotel e valores numéricos usados pelo processo
def extract_hotel_id(url_column: F.Column) -> F.Column:
    """Extrai o valor do parâmetro HotelId da URL, sem depender do restante dela."""
    hotel_id = F.trim(F.regexp_extract(url_column, r"(?i)[?&]HotelId=([^&#]+)", 1))
    return F.when(hotel_id != "", hotel_id)


def normalize_data(df: DataFrame) -> DataFrame:
    normalized = df
    text_columns = ["HOTEL", "DESTINO", "TIPO_DE_PRODUTO", "LINK_CVC_COM", "DESCONTO", "PAIS"]

    for expected in text_columns:
        actual = source_column(normalized, expected)
        normalized = normalized.withColumn(expected, clean_text(F.col(actual)))

    normalized = (
        normalized
        .withColumn("HOTEL_ID", extract_hotel_id(F.col("LINK_CVC_COM")))
        .withColumn(
            "PERIODO_INICIO",
            parse_date(F.col(source_column(normalized, "PERIODO_DE_EMBARQUE_INICIO_DD_MM_AAAA"))),
        )
        .withColumn(
            "PERIODO_FIM",
            parse_date(F.col(source_column(normalized, "PERIODO_DE_EMBARQUE_FIM_DD_MM_AAAA"))),
        )
        .withColumn("PRICE", parse_decimal(F.col(source_column(normalized, "DIARIA_POR_PESSOA_DE_R"))))
        .withColumn("SALE_PRICE", parse_decimal(F.col(source_column(normalized, "DIARIA_POR_PESSOA_POR_R"))))
    )
    return normalized

## 3. Expansão, HotelId e Product ID

O hotel é a entidade recomendada. O notebook extrai o parâmetro `HotelId` da URL e o usa como **Product ID** estável — sem destino, mês, tipo de quarto ou refeição.

A expansão mensal permanece como etapa de validação do período. Antes de gerar o Inventory, as variações do mesmo hotel são consolidadas: é mantida a oferta de menor `Sale Price`, usando `Price` como critério de desempate.

In [ ]:
# Cria uma oportunidade por mês; devolve também as linhas descartadas
def expand_opportunities_by_month(df: DataFrame) -> tuple[DataFrame, DataFrame]:
    invalid_reason = (
        F.when(F.col("PERIODO_INICIO").isNull(), "data de início nula ou inválida")
        .when(F.col("PERIODO_FIM").isNull(), "data de fim nula ou inválida")
        .when(F.col("PERIODO_FIM") < F.col("PERIODO_INICIO"), "data final anterior à inicial")
        .when(F.col("HOTEL_ID").isNull(), "HotelId ausente ou inválido na URL")
    )

    classified = df.withColumn("discard_reason", invalid_reason)
    discarded = classified.filter(F.col("discard_reason").isNotNull())
    valid = classified.filter(F.col("discard_reason").isNull())

    expanded = (
        valid.withColumn(
            "reference_month",
            F.explode(
                F.sequence(
                    F.trunc("PERIODO_INICIO", "month"),
                    F.trunc("PERIODO_FIM", "month"),
                    F.expr("interval 1 month"),
                )
            ),
        )
        .withColumn("REFERENCE_YYYY_MM", F.date_format("reference_month", "yyyy-MM"))
    )
    return expanded, discarded


def generate_product_id(df: DataFrame) -> DataFrame:
    """O hotel é o produto; o ID não varia por destino, mês, quarto ou refeição."""
    return df.withColumn("PRODUCT_ID", F.col("HOTEL_ID"))


def select_best_offer_by_hotel(df: DataFrame) -> DataFrame:
    """
    Mantém uma única oferta por hotel: a menor Sale Price.
    Price é usado como desempate quando Sale Price for igual ou estiver nulo.
    """
    hotel_window = Window.partitionBy("PRODUCT_ID").orderBy(
        F.col("SALE_PRICE").asc_nulls_last(),
        F.col("PRICE").asc_nulls_last(),
        F.col("LINK_CVC_COM").asc_nulls_last(),
    )

    return (
        df.withColumn("offer_rank", F.row_number().over(hotel_window))
        .filter(F.col("offer_rank") == 1)
        .drop("offer_rank")
    )

## 4. Montagem do layout de Recommendations

O DataFrame final segue exatamente o cabeçalho do modelo fornecido pela ferramenta, sem campos adicionais.

A categoria é fixa como `hotel_internacional`. A subcategoria é formada por país e cidade, em minúsculas e sem acentos, por exemplo: `estados_unidos_orlando`.

`Is_Del = 0` indica que o produto está ativo e não deve ser removido. `Is_Rec = 1` indica que ele pode participar das recomendações. O estoque é representado por `Count = 9999`, pois não há limite de unidades nesse contexto.

In [ ]:
def normalize_category_component(column: F.Column) -> F.Column:
    """Converte texto para minúsculas, sem acentos e separado por underscores."""
    text = clean_text(column)
    without_accents = F.translate(
        text,
        "ÁÀÂÃÄáàâãäÉÈÊËéèêëÍÌÎÏíìîïÓÒÔÕÖóòôõöÚÙÛÜúùûüÇçÑñ",
        "AAAAAaaaaaEEEEeeeeIIIIiiiiOOOOOoooooUUUUuuuuCcNn",
    )
    return F.regexp_replace(
        F.regexp_replace(F.lower(without_accents), r"[^a-z0-9]+", "_"),
        r"(^_+|_+$)",
        "",
    )


def category2_from_country_and_city() -> F.Column:
    country = normalize_category_component(F.col("PAIS"))
    city = normalize_category_component(F.col("DESTINO"))
    return F.when(country.isNotNull() & city.isNotNull(), F.concat_ws("_", country, city))


def build_recommendations_inventory(df: DataFrame) -> DataFrame:
    """
    Monta exatamente o layout fornecido pela ferramenta.
    Price usa Sale Price e recorre ao preço de tabela se não houver desconto.
    """
    inventory = df.select(
        F.col("PRODUCT_ID").alias("Pr_ID"),
        F.col("HOTEL").alias("Name"),
        F.lit("hotel_internacional").alias("Category1"),
        category2_from_country_and_city().alias("Category2"),
        F.coalesce(F.col("SALE_PRICE"), F.col("PRICE")).alias("Price"),
        F.lit("BRL").alias("Currency"),
        F.col("LINK_CVC_COM").alias("Page_URL"),
        F.lit(IMAGE_URL).alias("Image_URL"),
        F.lit(9999).cast("int").alias("Count"),
        F.lit("pt-BR").alias("Locale"),
        F.lit(0).cast("int").alias("Is_Del"),
        F.lit(1).cast("int").alias("Is_Rec"),
        F.lit(IMAGE_URL).alias("Thumbs_Image_URL"),
    )
    return inventory.select(*OUTPUT_COLUMNS)

## 5. Validações de qualidade

Antes de salvar, o notebook informa oportunidades descartadas e seus motivos, campos obrigatórios nulos, produtos por país e possíveis Product IDs duplicados.

Um `HotelId` ausente na URL também é motivo de descarte. Como o Inventory consolida uma única oferta por hotel, o Product ID deve ser único.

In [ ]:
def validate_inventory(inventory: DataFrame, discarded: DataFrame) -> None:
    print_step(6, "Validando dados do Inventory...")

    discarded_count = discarded.count()
    print(f"Oportunidades descartadas: {discarded_count}")

    if discarded_count > 0:
        print("Motivos de descarte:")
        discarded.groupBy("discard_reason").count().show(truncate=False)

    metrics = inventory.agg(
        F.count("*").alias("products"),
        F.countDistinct("Pr_ID").alias("unique_ids"),
        F.sum(F.when(F.col("Pr_ID").isNull() | (F.trim(F.col("Pr_ID")) == ""), 1).otherwise(0)).alias("null_product_ids"),
        F.sum(F.when(F.col("Name").isNull() | (F.trim(F.col("Name")) == ""), 1).otherwise(0)).alias("null_names"),
        F.sum(F.when(F.col("Page_URL").isNull() | (F.trim(F.col("Page_URL")) == ""), 1).otherwise(0)).alias("null_page_urls"),
        F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_prices"),
    ).first()

    print(f"Produtos gerados: {metrics['products']}")
    print(f"Product IDs únicos: {metrics['unique_ids']}")
    print(f"Registros com Product ID nulo: {metrics['null_product_ids'] or 0}")
    print(f"Registros com Name nulo: {metrics['null_names'] or 0}")
    print(f"Registros com Page URL nulo: {metrics['null_page_urls'] or 0}")
    print(f"Registros com Price nulo: {metrics['null_prices'] or 0}")

    print("Produtos por subcategoria:")
    inventory.groupBy("Category2").count().orderBy(F.desc("count")).show(100, truncate=False)

    duplicates = inventory.groupBy("Pr_ID").count().filter(F.col("count") > 1)
    duplicate_count = duplicates.count()
    print(f"Product IDs duplicados: {duplicate_count}")

    if duplicate_count:
        print("Exemplos de Product IDs duplicados:")
        duplicates.orderBy(F.desc("count"), "Pr_ID").show(20, truncate=False)
        if FAIL_ON_DUPLICATE_PRODUCT_IDS:
            raise ValueError("Existem Product IDs duplicados. Corrija a regra antes de publicar o arquivo.")

    if metrics["null_product_ids"] or metrics["null_prices"]:
        raise ValueError("Existem Product IDs ou preços nulos; o CSV não pode ser publicado.")

## 6. Prévia do CSV e geração do arquivo

Como esta é uma POC, o notebook exibe o Inventory também como texto CSV em uma caixa selecionável. Clique dentro da caixa, use **Ctrl+A** e depois **Ctrl+C**; no VS Code, cole o conteúdo em um arquivo com extensão `.csv`.

A prévia usa no máximo `CSV_PREVIEW_ROWS` linhas e traz apenas essa amostra para a memória do notebook. Quando as permissões forem liberadas, descomente `save_csv(inventory)` para gerar o arquivo no caminho configurado.

In [ ]:
def save_csv(inventory: DataFrame) -> None:
    """Escreve um CSV exatamente no caminho configurado em OUTPUT_PATH."""
    output_dir = OUTPUT_PATH.rsplit("/", 1)[0]
    file_name = OUTPUT_PATH.rsplit("/", 1)[1]
    temporary_dir = f"{output_dir}/_tmp_oracle_infinity_inventory"

    try:
        dbutils.fs.rm(temporary_dir, True)
    except Exception:
        pass

    (
        inventory.coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .option("quoteAll", True)
        .option("nullValue", "")
        .csv(temporary_dir)
    )
    part_file = next(item.path for item in dbutils.fs.ls(temporary_dir) if item.name.startswith("part-"))
    dbutils.fs.rm(OUTPUT_PATH, True)
    dbutils.fs.mv(part_file, f"{output_dir}/{file_name}")
    dbutils.fs.rm(temporary_dir, True)
    print_step(7, f"CSV gerado com sucesso: {OUTPUT_PATH}")


def display_inventory_as_csv(inventory: DataFrame, max_rows: int = CSV_PREVIEW_ROWS) -> None:
    """Exibe uma amostra do Inventory como CSV selecionável, somente para a POC."""
    preview = inventory.limit(max_rows).toPandas()
    csv_text = preview.to_csv(index=False, lineterminator="\n")

    print(f"Exibindo {len(preview)} linha(s) como CSV para cópia.")
    if len(preview) == max_rows:
        print(f"A prévia está limitada a {max_rows} linha(s). Aumente CSV_PREVIEW_ROWS se necessário.")

    displayHTML(
        f'<textarea style="width:100%; height:500px; font-family:monospace; white-space:pre;" '
        f'readonly>{html.escape(csv_text)}</textarea>'
    )


def main() -> None:
    print_step(1, "Iniciando processo...")
    source = load_data()
    normalized = normalize_data(source)
    print_step(4, "Processando períodos e quebrando oportunidades por mês...")
    expanded, discarded = expand_opportunities_by_month(normalized)

    offers_with_product_id = generate_product_id(expanded)
    selected_offers = select_best_offer_by_hotel(offers_with_product_id)
    inventory = build_recommendations_inventory(selected_offers)

    print_step(5, f"Produtos gerados: {inventory.count()}")
    validate_inventory(inventory, discarded)
    print("Exemplos do resultado final:")
    inventory.show(10, truncate=False)
    display_inventory_as_csv(inventory)
    # save_csv(inventory)


main()